<a href="https://colab.research.google.com/github/Mareza23/Documents/blob/master/ch02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chapter 2: Pre-Model Workflow and Data Preprocessing

Kualitas data sangat menentukan kualitas model *machine learning*. Prinsip ini sering disebut *garbage in, garbage out*, yang berarti data dengan kualitas buruk dapat menghasilkan model dengan performa yang kurang baik. Oleh karena itu, sebelum data digunakan untuk melatih model, diperlukan tahap *data preprocessing* untuk mempersiapkan data agar lebih sesuai untuk proses pembelajaran.

Beberapa masalah yang umum ditemukan dalam tahap preprocessing beserta tools yang tersedia di scikit-learn adalah sebagai berikut.

| Masalah                        | Tools di scikit-learn                                                  | Fungsi                                                                                                |
| ------------------------------ | ---------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- |
| Data hilang (*missing values*) | `SimpleImputer`, `KNNImputer`, `IterativeImputer`                      | Menangani nilai yang kosong pada dataset.                                                             |
| Skala fitur berbeda            | `StandardScaler`, `MinMaxScaler`, `Normalizer`, `RobustScaler`         | Menyesuaikan skala fitur sesuai kebutuhan model.                                                      |
| Fitur kategorikal              | `OneHotEncoder`, `OrdinalEncoder`, `LabelEncoder`, `ColumnTransformer` | Mengolah data kategorikal agar dapat digunakan dalam pemodelan.                                       |
| Risiko *data leakage*          | `Pipeline`                                                             | Mengatur tahapan preprocessing dan pemodelan agar data pelatihan dan pengujian diproses dengan benar. |
| Fitur kurang informatif        | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel`     | Membantu membentuk, mengelompokkan, atau memilih fitur untuk pemodelan.                               |

**Petunjuk penggunaan:** Jalankan setiap sel kode dari atas ke bawah. Setiap bagian perlu dilengkapi dengan penjelasan konsep, fungsi kode, dan interpretasi hasil yang diperoleh.


In [1]:
import numpy as np
import pandas as pd
import sklearn
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

scikit-learn: 1.6.1
NumPy       : 2.1.3
pandas      : 2.2.3


**Penjelasan:** sel ini memuat library dan mengatur tampilan pandas (maksimal 15 kolom, 4 angka desimal) supaya tabel mudah dibaca. Mengecek versi berguna karena perilaku beberapa fungsi bisa berbeda antar versi.

**1. Mengapa Preprocessing Penting?**
Masalah data yang paling sering muncul:  

1.   **Data hilang**: sebagian besar estimator scikit-learn menolak input yang berisi NaN.
2.   **Outlier**: nilai ekstrem bisa menarik parameter model jauh dari pola sebenarnya.
3. **Variabel kategorikal**: model hanya memahami angka, jadi teks harus dikodekan.
4. **Skala fitur berbeda**: fitur dengan angka besar mendominasi fitur berangka kecil pada algoritma berbasis jarak atau gradien.
5. **Data leakage:** informasi dari data uji ikut dipakai saat preprocessing sehingga skor evaluasi terlihat lebih bagus daripada kenyataan.

Untuk melihat dampak nyata poin 4, kita bandingkan KNN pada dataset wine dengan dan tanpa penskalaan.


In [2]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xw, yw = load_wine(return_X_y=True, as_frame=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(
    Xw, yw, test_size=0.3, random_state=42, stratify=yw
)

# Kisaran nilai tiap fitur (sangat berbeda-beda)
print(Xw.describe().loc[["min", "max"]].T.head(6))

knn_raw = KNeighborsClassifier().fit(Xw_tr, yw_tr)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier()).fit(Xw_tr, yw_tr)

print(f"\nAkurasi KNN tanpa scaling : {knn_raw.score(Xw_te, yw_te):.4f}")
print(f"Akurasi KNN dengan scaling: {knn_scaled.score(Xw_te, yw_te):.4f}")

                      min      max
alcohol           11.0300  14.8300
malic_acid         0.7400   5.8000
ash                1.3600   3.2300
alcalinity_of_ash 10.6000  30.0000
magnesium         70.0000 162.0000
total_phenols      0.9800   3.8800

Akurasi KNN tanpa scaling : 0.7222
Akurasi KNN dengan scaling: 0.9444


**Penjelasan hasil**: pada dataset ini fitur proline bernilai ratusan sampai ribuan, sedangkan fitur lain hanya berkisar puluhan atau di bawah 10. KNN menghitung jarak, sehingga tanpa scaling jarak hampir sepenuhnya ditentukan oleh proline. Setelah diskalakan, semua fitur berkontribusi seimbang dan akurasinya naik jelas. Ini alasan utama scaling dilakukan pada algoritma berbasis jarak.

## 2. **Menangani Data Hilang (*Missing Data*)**

Tiga metode imputasi yang tersedia di scikit-learn:

| **Metode**         | **Cara kerja**                                                                                            | **Cocok untuk**                                           |
| :----------------- | :-------------------------------------------------------------------------------------------------------- | :-------------------------------------------------------- |
| `SimpleImputer`    | Mengisi nilai yang hilang menggunakan mean, median, modus, atau konstanta.                                | Penanganan data hilang secara sederhana dan cepat.        |
| `KNNImputer`       | Mengisi nilai yang hilang berdasarkan nilai dari tetangga terdekat.                                       | Data numerik yang memiliki pola atau kemiripan antardata. |
| `IterativeImputer` | Memperkirakan nilai yang hilang dengan memodelkan setiap fitur berdasarkan fitur lainnya secara berulang. | Data yang memiliki hubungan antarfitur yang cukup kuat.   |
